# Подготовка

## Импорты

In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

from src.dataset import DatasetStore
from src.runner import ExperimentRunner
from src.pipeline import Pipeline
from src.preprocessors import *
from src.models import *
from src.validators import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [3]:
df, schema = DatasetStore.load(
    "data/processed/dataset.csv",
    "data/processed/dataset.schema.json",
    include_missing=False,
)
feature_info = DatasetStore.get_feature_info(
    schema,
    include_missing=False,
)

In [4]:
columns = ['bled cut', 'albumin']
for col in columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [5]:
df["diagnosis"] = df["diagnosis"].replace(6, 5)

y = LabelEncoder().fit_transform(df['diagnosis'].values)
X = df.drop(columns=['diagnosis'])

class_labels = [0, 1]

## Инициализация компонентов

In [6]:
onehot_prep = OneHotPreprocessor(
    categorical_features=feature_info['categorical_features'],
    ordinal_features=feature_info['ordinal_features'],
    ordinal_categories=feature_info['ordinal_categories'],
    missing_value=feature_info['missing_value'],
)

In [7]:
validator = RepeatedStratifiedKFoldValidator(
    n_splits=3, n_repeats=10, random_state=42,
)

In [8]:
runner = ExperimentRunner(results_dir='_analys')

# Обучение

In [9]:
logreg = LogisticRegressionModel(
    C=0.5,
    solver="saga",
    max_iter=300,
    class_weight="balanced",
)

## Все признаки

In [10]:
runner.run_single(
    name="LogReg",
    preprocessor=onehot_prep,
    model=logreg,
    validator=validator,
    X=X, y=y,
    class_labels=class_labels,
    save_pipeline=True,
)


Запуск эксперимента: LogReg
💾 Detailed сохранён: _analys\LogReg\detailed.json
💾 Pipeline сохранён: _analys\LogReg\pipeline.joblib


{'config': {'preprocessor_params': {'categorical_features': ['Sex',
    'Ds naprav',
    'soput disease',
    'compens',
    'pain',
    'ogirenie',
    't',
    'encef',
    'bled cut',
    'mramor cut',
    'icterus',
    'hipermia face',
    'AD low',
    'Kollaps',
    'meteorism',
    'pain of adbomin',
    'rigid of abdomi',
    's. peritonit',
    'Er',
    'Hb',
    'amilase',
    'Ur',
    'gemolis',
    'ECG',
    'LS/cen gid',
    'LS steat'],
   'exclude_features': None,
   'missing_value': None,
   'ordinal_categories': {'Age': ['до 20 лет',
     '21-30',
     '31-40',
     '41-50',
     '51-60',
     'старше 60 лет'],
    'time disease': ['более 24 часов', 'до 24 часов'],
    'chr pancr': ['есть', 'приступов не было'],
    'intensiv': ['умеренная', 'интенсивная'],
    'rvota': ['не было', 'была'],
    'cofe gugha': ['не было', 'была'],
    'slabost': ['не было', 'были'],
    'Ps': ['менее 80', '80-99', '100-120', 'более 120'],
    'CHDD': ['16-20', '21-26', 'более 26'],
 

## Удалены коррелирующие признаки

In [11]:
features = ["compens", "bil", "Hb"]
df_removed = df.drop(columns=features)
X_removed = df_removed.drop(columns=['diagnosis'])

In [12]:
runner.run_single(
    name="LogRegRemovedFeatures",
    preprocessor=onehot_prep,
    model=logreg,
    validator=validator,
    X=X_removed, y=y,
    class_labels=class_labels,
    save_pipeline=True,
)


Запуск эксперимента: LogRegRemovedFeatures
💾 Detailed сохранён: _analys\LogRegRemovedFeatures\detailed.json
💾 Pipeline сохранён: _analys\LogRegRemovedFeatures\pipeline.joblib


{'config': {'preprocessor_params': {'categorical_features': ['Sex',
    'Ds naprav',
    'soput disease',
    'compens',
    'pain',
    'ogirenie',
    't',
    'encef',
    'bled cut',
    'mramor cut',
    'icterus',
    'hipermia face',
    'AD low',
    'Kollaps',
    'meteorism',
    'pain of adbomin',
    'rigid of abdomi',
    's. peritonit',
    'Er',
    'Hb',
    'amilase',
    'Ur',
    'gemolis',
    'ECG',
    'LS/cen gid',
    'LS steat'],
   'exclude_features': None,
   'missing_value': None,
   'ordinal_categories': {'Age': ['до 20 лет',
     '21-30',
     '31-40',
     '41-50',
     '51-60',
     'старше 60 лет'],
    'time disease': ['более 24 часов', 'до 24 часов'],
    'chr pancr': ['есть', 'приступов не было'],
    'intensiv': ['умеренная', 'интенсивная'],
    'rvota': ['не было', 'была'],
    'cofe gugha': ['не было', 'была'],
    'slabost': ['не было', 'были'],
    'Ps': ['менее 80', '80-99', '100-120', 'более 120'],
    'CHDD': ['16-20', '21-26', 'более 26'],
 